# Generate Next-State Extrapolation Data

This Generate notebook computes and caches the data consumed by `Visualize-extrapolation-scaling.ipynb`. The companion Visualize notebook shows how normalized expected-value error for predicting the next quantized state scales with context length across the main numerical input families using Llama-3.2-3B, as presented in the appendix **Main Numerical Input Suite and Next-State Extrapolation**.


Set imports, constants, output paths, sweep axes, and random seeds.


In [ ]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import gc
import random
from pathlib import Path

import numpy as np
import torch
from scipy.integrate import solve_ivp
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Model and sweep configuration
SEED = 42
ENSEMBLE_SEED = SEED
MODEL_NAME = "meta-llama/Llama-3.2-3B"
NUM_ENSEMBLE_RUNS = 20
TOTAL_SEQUENCE_STATES = 1000

# Quantized-state range and normalization for absolute prediction error
SCALING_QUANTIZED_STATE_MIN = 50
SCALING_QUANTIZED_STATE_MAX = 950
SCALING_ERROR_NORMALIZATION = float(SCALING_QUANTIZED_STATE_MAX - SCALING_QUANTIZED_STATE_MIN)
SCALING_ERROR_LOG_FLOOR = 1e-6

# Save resumable progress separately from the compact visualization cache
CACHE_PATH = Path("cache/extrapolation_scaling_data.npz")
CHECKPOINT_PATH = Path("cache/extrapolation_scaling_checkpoint.npz")
RESUME_FROM_CHECKPOINT = True
CHECKPOINT_EVERY_N_TASKS = 1
NPZ_SCHEMA_VERSION = "extrapolation_scaling_minimal"
CHECKPOINT_SCHEMA_VERSION = NPZ_SCHEMA_VERSION + "_checkpoint"
# Include the model's usual special tokens, such as Llama's leading BOS, in each prompt
TOKENIZATION_ADD_SPECIAL_TOKENS = True

# Context-length sweep axes
context_fracs = np.asarray([i / 100 for i in range(2, 100, 2)], dtype=np.float64)
context_percent = np.asarray([int(round(100 * frac)) for frac in context_fracs], dtype=np.int64)
run_ids = np.arange(NUM_ENSEMBLE_RUNS, dtype=np.int64)

data_order = np.asarray([
    "Constant sampled",
    "Logistic r=3.2",
    "Logistic r=3.5",
    "Logistic r=3.55",
    "Logistic r=3.566",
    "Logistic r=3.5692",
    "Logistic r=3.5698",
    "Logistic r=3.9",
    "Logistic r=4.0",
    "Lorenz",
], dtype=str)

# Seed data generation and model inference
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


Define the Lorenz system and shared sequence-length conventions.


In [ ]:
# Lorenz parameters and sampling grid
LORENZ_SIGMA = 10.0
LORENZ_RHO = 28.0
LORENZ_BETA = 8.0 / 3.0
LORENZ_ARGS = (LORENZ_SIGMA, LORENZ_RHO, LORENZ_BETA)
LORENZ_T_SPAN = (0.0, 100.0)

N = TOTAL_SEQUENCE_STATES
lorenz_t_eval = np.linspace(LORENZ_T_SPAN[0], LORENZ_T_SPAN[1], N)


def lorenz(t, y, sigma, rho, beta):
    return [
        sigma * (y[1] - y[0]),
        y[0] * (rho - y[2]) - y[1],
        y[0] * y[1] - beta * y[2],
    ]


Define sequence generators and prompt serialization helpers.


In [ ]:
# Periodic logistic-map regimes and burn-in controls
periodic_logistic_periods = {
    3.2: 2,
    3.5: 4,
    3.55: 8,
    3.566: 16,
    3.5692: 32,
    3.5698: 64,
}
logistic_long_n = 20000
logistic_settle_tol = 1e-10


def logistic_sequence(r, x0, n):
    x = np.empty(n, dtype=float)
    x[0] = x0
    for i in range(1, n):
        x[i] = r * x[i - 1] * (1 - x[i - 1])
    return x


def settle_time_float(x, p, tol=1e-10):
    x = np.asarray(x, dtype=float)
    if p is None or p <= 0 or p >= x.size:
        return None
    # Find the first index after which lag-p error remains below tolerance
    err = np.abs(x[p:] - x[:-p])
    suffix_max = np.maximum.accumulate(err[::-1])[::-1]
    hits = np.flatnonzero(suffix_max < tol)
    return int(hits[0]) if hits.size else None


# Scale each trajectory independently into the 050..950 token range
def quantize_to_state_tokens(values):
    values = np.asarray(values, dtype=float)
    value_min = float(values.min())
    value_max = float(values.max())
    if value_max == value_min:
        return np.full(values.shape[0], 500, dtype=int)
    return np.round(
        SCALING_QUANTIZED_STATE_MIN
        + (values - value_min) / (value_max - value_min) * SCALING_ERROR_NORMALIZATION
    ).astype(int)


def build_burned_in_logistic_window(
    r,
    x0,
    n_window,
    expected_period,
    n_long=logistic_long_n,
    tol=logistic_settle_tol,
    phase_offset=0,
):
    # Discard transients and select a phase-aligned periodic window
    phase_offset = int(phase_offset) % int(expected_period)
    x_long = logistic_sequence(r, x0, n_long)
    t_settle = settle_time_float(x_long, expected_period, tol=tol)
    if t_settle is None:
        raise ValueError(f"Could not find settle time for r={r} with expected period {expected_period}.")

    start_idx = int(t_settle) + int(phase_offset)
    if start_idx + n_window > x_long.size:
        raise ValueError(f"Need a longer trajectory for r={r}; start index {start_idx} leaves fewer than {n_window} states.")

    return np.asarray(x_long[start_idx:start_idx + n_window], dtype=float)


# Reserve one state beyond the context as the next-state target
def _context_len_from_frac(sequence_len, context_frac):
    ctx_len = max(1, int(round(float(context_frac) * (int(sequence_len) - 1))))
    return min(ctx_len, int(sequence_len) - 1)


# Emit one zero-padded number token and one comma token per state
def _serialized_prompt_from_states(states):
    states = np.asarray(states, dtype=int)
    return ",".join(f"{int(x):03d}" for x in states) + ","


# Precompute state counts and corresponding post-BOS content-token counts
context_state_counts = np.asarray(
    [_context_len_from_frac(TOTAL_SEQUENCE_STATES, frac) for frac in context_fracs],
    dtype=np.int64,
)
context_token_counts = 2 * context_state_counts


Build the constant, logistic, and Lorenz ensemble inputs.


In [ ]:
# Share each replicate's x0 across all input families
periodic_rs = list(periodic_logistic_periods.keys())
chaotic_rs = [3.9, 4.0]

ensemble_rng = np.random.default_rng(ENSEMBLE_SEED)
phase_rng = np.random.default_rng(ENSEMBLE_SEED + 1)
x0_samples = ensemble_rng.uniform(0.3, 0.7, size=NUM_ENSEMBLE_RUNS)
ensemble_inputs = {}

for rep_id, x0_rep in zip(run_ids, x0_samples):
    x0_rep = float(x0_rep)
    rep_inputs = {}

    # Repeat one sampled token throughout each constant realization
    constant_token = int(
        np.round(SCALING_QUANTIZED_STATE_MIN + SCALING_ERROR_NORMALIZATION * x0_rep)
    )
    rep_inputs["Constant sampled"] = np.full(N, constant_token, dtype=int)

    # Use settled periodic windows with randomized phase
    for r in periodic_rs:
        expected_period = periodic_logistic_periods[r]
        phase_offset = int(phase_rng.integers(0, expected_period))
        values = build_burned_in_logistic_window(
            r,
            x0=x0_rep,
            n_window=N,
            expected_period=expected_period,
            phase_offset=phase_offset,
        )
        rep_inputs[f"Logistic r={r}"] = quantize_to_state_tokens(values)

    # Start chaotic trajectories directly from the shared sampled x0
    for r in chaotic_rs:
        values = logistic_sequence(r, x0_rep, N)
        rep_inputs[f"Logistic r={r}"] = quantize_to_state_tokens(values)

    # Perturb Lorenz x0 and quantize the resulting x-coordinate
    lorenz_initial_conditions = [float(-8.0 + (x0_rep - 0.5)), 7.0, 27.0]
    sol = solve_ivp(
        lorenz,
        LORENZ_T_SPAN,
        lorenz_initial_conditions,
        args=LORENZ_ARGS,
        t_eval=lorenz_t_eval,
    )
    if not sol.success:
        raise RuntimeError(f"Lorenz solve failed for rep {int(rep_id)}: {sol.message}")
    rep_inputs["Lorenz"] = quantize_to_state_tokens(sol.y[0])

    missing_inputs = sorted(set(data_order.astype(str)).difference(rep_inputs.keys()))
    if missing_inputs:
        raise ValueError(f"Missing inputs for rep {int(rep_id)}: {missing_inputs}")
    ensemble_inputs[int(rep_id)] = rep_inputs


Load the tokenizer, validate numeric-state tokenization, and define expected-value prediction.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)


# Exclude an optional leading BOS when indexing serialized content
def _context_token_start(input_ids, bos_token_id=None):
    if input_ids is None or bos_token_id is None:
        return 0
    if isinstance(input_ids, torch.Tensor):
        first_id = int(input_ids.flatten()[0].item())
    else:
        first_id = int(np.asarray(input_ids).flatten()[0])
    return 1 if first_id == int(bos_token_id) else 0


# Map every zero-padded state token to its numeric value
def build_valid_number_token_lookup(tokenizer):
    token_to_int = {}
    for value in range(1000):
        token_str = f"{value:03d}"
        token_ids = tokenizer(token_str, add_special_tokens=False).input_ids
        if len(token_ids) != 1:
            raise ValueError(f"Expected one token for {token_str!r}, got {token_ids}.")
        token_to_int[int(token_ids[0])] = value
    return token_to_int


# Require one token per state and comma for exact context sizing
comma_token_ids = tokenizer(",", add_special_tokens=False).input_ids
if len(comma_token_ids) != 1:
    raise ValueError(f"Expected one comma token, got {comma_token_ids}.")
COMMA_TOKEN_ID = int(comma_token_ids[0])

numeric_token_to_int = build_valid_number_token_lookup(tokenizer)
numeric_token_ids = list(numeric_token_to_int.keys())
if len(numeric_token_ids) != 1000:
    raise ValueError(f"Expected 1000 valid numeric tokens, got {len(numeric_token_ids)}.")

numeric_token_values = [numeric_token_to_int[token_id] for token_id in numeric_token_ids]

# Precompute the restricted numeric vocabulary used for expectation values
numeric_token_ids_tensor = torch.tensor(numeric_token_ids, dtype=torch.long)
numeric_token_values_tensor = torch.tensor(numeric_token_values, dtype=torch.float32)


def predict_context_sweep_expected_values_from_long_prompt(sequence, model, tokenizer):
    # Tokenize the longest prompt once and reuse its context boundaries
    seq_arr = np.asarray(sequence, dtype=int)
    max_ctx_len = int(context_state_counts.max())
    prompt = _serialized_prompt_from_states(seq_arr[:max_ctx_len])

    device = next(model.parameters()).device
    inputs = tokenizer(
        prompt,
        add_special_tokens=TOKENIZATION_ADD_SPECIAL_TOKENS,
        return_tensors="pt",
    ).to(device)
    token_start = _context_token_start(inputs.input_ids[0], bos_token_id=tokenizer.bos_token_id)
    input_len = int(inputs.input_ids.shape[1])
    # Verify that serialization remains exactly number, comma per state
    non_bos_token_count = input_len - int(token_start)
    if non_bos_token_count != int(2 * max_ctx_len):
        raise ValueError(f"Expected {2 * max_ctx_len} non-BOS prompt tokens, got {non_bos_token_count}.")

    # Read next-token logits after the comma ending each requested context
    comma_positions = [int(token_start + 2 * int(ctx_len) - 1) for ctx_len in context_state_counts]
    for comma_pos in comma_positions:
        if int(inputs.input_ids[0, comma_pos].item()) != COMMA_TOKEN_ID:
            raise ValueError(f"Expected comma token id {COMMA_TOKEN_ID} at position {comma_pos}.")

    comma_positions_tensor = torch.tensor(comma_positions, dtype=torch.long, device=device)
    with torch.no_grad():
        outputs = model(**inputs, return_dict=True, use_cache=False)
        logits_at_commas = outputs.logits[0].index_select(dim=0, index=comma_positions_tensor)
        token_ids = numeric_token_ids_tensor.to(logits_at_commas.device)
        token_values = numeric_token_values_tensor.to(logits_at_commas.device)
        # Renormalize over valid numeric tokens and compute the expected state
        selected_logits = logits_at_commas.index_select(dim=-1, index=token_ids)
        probs = torch.softmax(selected_logits.float(), dim=-1)
        expected_values = (probs * token_values).sum(dim=-1).detach().cpu().numpy()

    del outputs
    return {
        "seq_len": context_token_counts.copy(),
        "true_next_state": seq_arr[context_state_counts].astype(np.int64),
        "predicted_value": expected_values.astype(np.float64),
    }


Create, validate, load, and save checkpoint arrays for resumable sweeps.


In [ ]:
def _expected_result_shapes():
    return (len(data_order), len(run_ids), len(context_fracs))


def _initialize_result_arrays():
    # Retain one error per input family, realization, and context length
    base_shape = _expected_result_shapes()
    return {
        "normalized_abs_error": np.full(base_shape, np.nan, dtype=np.float64),
        "completed_mask": np.zeros(base_shape, dtype=bool),
    }


def _npz_scalar_to_str(data, key):
    return str(np.asarray(data[key]).item())


def _validate_checkpoint_compatibility(data):
    # Accept only checkpoints matching the model, sweep, and schema
    if _npz_scalar_to_str(data, "checkpoint_schema_version") != CHECKPOINT_SCHEMA_VERSION:
        raise ValueError("Checkpoint schema version mismatch.")
    if _npz_scalar_to_str(data, "npz_schema_version") != NPZ_SCHEMA_VERSION:
        raise ValueError("Checkpoint final schema version mismatch.")
    if _npz_scalar_to_str(data, "model_name") != MODEL_NAME:
        raise ValueError("Checkpoint model_name mismatch.")
    if int(np.asarray(data["total_sequence_states"]).item()) != TOTAL_SEQUENCE_STATES:
        raise ValueError("Checkpoint sequence length mismatch.")
    if not np.array_equal(data["data_order"].astype(str), data_order.astype(str)):
        raise ValueError("Checkpoint data_order mismatch.")
    if not np.array_equal(data["run_ids"].astype(np.int64), run_ids.astype(np.int64)):
        raise ValueError("Checkpoint run_ids mismatch.")
    if not np.allclose(data["context_fracs"].astype(np.float64), context_fracs.astype(np.float64)):
        raise ValueError("Checkpoint context_fracs mismatch.")
    if not np.array_equal(data["context_token_counts"].astype(np.int64), context_token_counts.astype(np.int64)):
        raise ValueError("Checkpoint context_token_counts mismatch.")

    expected_shape = _expected_result_shapes()
    for key in ["normalized_abs_error", "completed_mask"]:
        if data[key].shape != expected_shape:
            raise ValueError(f"Checkpoint {key} shape {data[key].shape}, expected {expected_shape}.")


def load_or_initialize_checkpoint():
    if RESUME_FROM_CHECKPOINT and CHECKPOINT_PATH.exists():
        with np.load(CHECKPOINT_PATH, allow_pickle=False) as data:
            _validate_checkpoint_compatibility(data)
            arrays = {
                "normalized_abs_error": data["normalized_abs_error"].astype(np.float64),
                "completed_mask": data["completed_mask"].astype(bool),
            }
        return arrays
    return _initialize_result_arrays()


def save_checkpoint(arrays):
    # Save progress together with all compatibility metadata
    CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        CHECKPOINT_PATH,
        data_order=data_order,
        context_fracs=context_fracs,
        context_percent=context_percent,
        context_state_counts=context_state_counts,
        context_token_counts=context_token_counts,
        run_ids=run_ids,
        model_name=np.asarray(MODEL_NAME),
        total_sequence_states=np.asarray(TOTAL_SEQUENCE_STATES, dtype=np.int64),
        npz_schema_version=np.asarray(NPZ_SCHEMA_VERSION),
        checkpoint_schema_version=np.asarray(CHECKPOINT_SCHEMA_VERSION),
        normalized_abs_error=arrays["normalized_abs_error"],
        completed_mask=arrays["completed_mask"],
    )


Load the model for expected-value next-token prediction.


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
)


Run or resume the ensemble expected-value extrapolation sweep.


In [ ]:
arrays = load_or_initialize_checkpoint()
completed_since_checkpoint = 0
checkpoint_every = max(1, int(CHECKPOINT_EVERY_N_TASKS))

# Task grid: input family x realization; one model pass covers all contexts
tasks = [
    (input_axis_idx, str(data_name), run_axis_idx, int(rep_id))
    for input_axis_idx, data_name in enumerate(data_order)
    for run_axis_idx, rep_id in enumerate(run_ids)
]

for input_axis_idx, data_name, run_axis_idx, rep_id in tqdm(
    tasks,
    desc="Ensemble extrapolation sweep",
    miniters=checkpoint_every,
):
    # Skip input realizations already completed in a compatible checkpoint
    if arrays["completed_mask"][input_axis_idx, run_axis_idx].all():
        continue

    # Evaluate all context boundaries from one serialized longest prompt
    sequence = ensemble_inputs[int(rep_id)][data_name]
    prediction = predict_context_sweep_expected_values_from_long_prompt(sequence, model, tokenizer)
    if not np.array_equal(prediction["seq_len"].astype(np.int64), context_token_counts):
        raise ValueError("Prediction sequence lengths do not match the configured context token counts.")

    # Store normalized expected-value errors across context lengths
    true_next_state = prediction["true_next_state"].astype(np.float64)
    predicted_value = prediction["predicted_value"].astype(np.float64)
    abs_error = np.abs(predicted_value - true_next_state)
    arrays["normalized_abs_error"][input_axis_idx, run_axis_idx, :] = (
        abs_error / SCALING_ERROR_NORMALIZATION
    )
    arrays["completed_mask"][input_axis_idx, run_axis_idx, :] = True
    completed_since_checkpoint += 1
    del prediction, true_next_state, predicted_value, abs_error
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if completed_since_checkpoint >= checkpoint_every:
        save_checkpoint(arrays)
        completed_since_checkpoint = 0

if completed_since_checkpoint > 0 or not CHECKPOINT_PATH.exists():
    save_checkpoint(arrays)


Validate completed arrays and save the compact visualization cache.


In [ ]:
def _validate_completed_arrays():
    # Reject incomplete, malformed, or invalid errors before export
    if not np.all(arrays["completed_mask"]):
        bad = np.argwhere(~arrays["completed_mask"])[:10]
        raise ValueError(f"Not all tasks are complete; first incomplete indices: {bad}.")

    normalized_abs_error = arrays["normalized_abs_error"]
    if normalized_abs_error.shape != _expected_result_shapes():
        raise ValueError(f"normalized_abs_error shape {normalized_abs_error.shape}, expected {_expected_result_shapes()}.")
    if not np.isfinite(normalized_abs_error).all():
        bad = np.argwhere(~np.isfinite(normalized_abs_error))[:10]
        raise ValueError(f"normalized_abs_error contains nonfinite values; first bad indices: {bad}.")
    if not np.all(normalized_abs_error >= 0.0):
        bad = np.argwhere(normalized_abs_error < 0.0)[:10]
        raise ValueError(f"normalized_abs_error contains negative values; first bad indices: {bad}.")


_validate_completed_arrays()

# Flatten input x realization x context into table-like cache columns
input_axis, run_axis, context_axis = np.meshgrid(
    np.arange(len(data_order), dtype=np.int64),
    np.arange(len(run_ids), dtype=np.int64),
    np.arange(len(context_fracs), dtype=np.int64),
    indexing="ij",
)

# Save the arrays and metadata needed by the companion visualization
CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    CACHE_PATH,
    rep_id=run_ids[run_axis.ravel()].astype(np.int64),
    data_name=data_order[input_axis.ravel()].astype(str),
    seq_len=context_token_counts[context_axis.ravel()].astype(np.int64),
    normalized_abs_error=arrays["normalized_abs_error"].ravel().astype(np.float64),
    data_order=data_order.astype(str),
    scaling_error_log_floor=np.asarray(SCALING_ERROR_LOG_FLOOR, dtype=np.float64),
    scaling_error_normalization=np.asarray(SCALING_ERROR_NORMALIZATION, dtype=np.float64),
    quantized_state_min=np.asarray(SCALING_QUANTIZED_STATE_MIN, dtype=np.int64),
    quantized_state_max=np.asarray(SCALING_QUANTIZED_STATE_MAX, dtype=np.int64),
)
